<a href="https://colab.research.google.com/github/ttlttk8161/ML_practice/blob/MNIST_branch/MNIST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torchvision.datasets as dsets
import torchvision.transforms as transforms
import torch.nn.init
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [16]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [17]:
from pickle import TRUE
torch.manual_seed(777)

if device == 'cuda':
  torch.cuda.manual_seed_all(777)

learning_rate = 0.001
training_epochs = 15
batch_size = 100

mnist_train = dsets.MNIST(root='MNIST_data/', train=True, transform=transforms.ToTensor(), download=True)

mnist_test = dsets.MNIST(root='MNIST_data/', train=False, transform=transforms.ToTensor(), download=True)

data_loader = torch.utils.data.DataLoader(dataset=mnist_train, batch_size=batch_size, shuffle=True, drop_last=True)

100%|██████████| 9.91M/9.91M [00:00<00:00, 18.8MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 523kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.70MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 13.7MB/s]


In [18]:
class CNN(torch.nn.Module):
  def __init__(self) -> None:
    super(CNN, self).__init__()
    self.layer1 = torch.nn.Sequential(torch.nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1), torch.nn.ReLU(), torch.nn.MaxPool2d(kernel_size=2, stride=2))
    self.layer2 = torch.nn.Sequential(torch.nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1), torch.nn.ReLU(), torch.nn.MaxPool2d(kernel_size=2, stride=2))

    self.fc = torch.nn.Linear(7*7*64, 10, bias=True)

    torch.nn.init.xavier_uniform_(self.fc.weight)

  def forward(self, x):
    out = self.layer1(x)
    out = self.layer2(out)
    out = out.view(out.size(0), -1)
    return out


In [19]:
model = CNN().to(device)

In [21]:
from functools import total_ordering
criterion = torch.nn.CrossEntropyLoss().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr= learning_rate)

total_batch = len(data_loader)
print('총 배치의 수 : {}'.format(total_batch))

for epoch in range(training_epochs):
  avg_cost = 0

  for X, Y in data_loader:
    X = X.to(device)
    Y = Y.to(device)

    optimizer.zero_grad()
    hypothesis = model(X)
    cost = criterion(hypothesis, Y)
    cost.backward()
    optimizer.step()

    avg_cost += cost / total_batch

  print('[Epoch: {:>4}] cost = {:>.9}'.format(epoch + 1, avg_cost))

  with torch.no_grad():
    X_test = mnist_test.test_data.view(len(mnist_test), 1, 28, 28).float().to(device)
    Y_test = mnist_test.test_labels.to(device)

    predicition = model(X_test)
    correct_prediction = torch.argmax(predicition, 1) == Y_test
    accuracy = correct_prediction.float().mean()
    print('Accuracy:', accuracy.item())

총 배치의 수 : 600
[Epoch:    1] cost = 2.784235
Accuracy: 0.13519999384880066


/usr/local/lib/python3.12/dist-packages/torchvision/datasets/mnist.py:81: UserWarning: test_data has been renamed data
  warnings.warn("test_data has been renamed data")
/usr/local/lib/python3.12/dist-packages/torchvision/datasets/mnist.py:71: UserWarning: test_labels has been renamed targets
  warnings.warn("test_labels has been renamed targets")


[Epoch:    2] cost = 2.67160845
Accuracy: 0.13249999284744263
[Epoch:    3] cost = 2.65485263
Accuracy: 0.13079999387264252
[Epoch:    4] cost = 2.64621425
Accuracy: 0.13339999318122864
[Epoch:    5] cost = 2.64116073
Accuracy: 0.12950000166893005
[Epoch:    6] cost = 2.6366396
Accuracy: 0.13159999251365662
[Epoch:    7] cost = 2.63335347
Accuracy: 0.12999999523162842
[Epoch:    8] cost = 2.62942576
Accuracy: 0.12929999828338623
[Epoch:    9] cost = 2.62612057
Accuracy: 0.13130000233650208
[Epoch:   10] cost = 2.62375021
Accuracy: 0.13120000064373016
[Epoch:   11] cost = 2.62145615
Accuracy: 0.13030000030994415
[Epoch:   12] cost = 2.61908412
Accuracy: 0.13339999318122864
[Epoch:   13] cost = 2.61720061
Accuracy: 0.13319998979568481
[Epoch:   14] cost = 2.61578798
Accuracy: 0.13300000131130219
[Epoch:   15] cost = 2.6144557
Accuracy: 0.13040000200271606
